### SQLの勉強

In [1]:
import pandas as pd
from pandasql import sqldf
import sqlite3
import numpy as np

In [2]:
df = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35]
})
df

,id,name,age
0,1,Alice,25
1,2,Bob,30
2,3,Charlie,35


### フィルタ

In [3]:
conn = sqlite3.connect(':memory:')
df.to_sql('df', conn, index=False, if_exists='replace')

query = """SELECT * 
            FROM df 
            WHERE age > 30"""#whereの引数は特にカラム名に""で囲う必要はない。文字型だった場合は指定したカラムの要素のみに""で囲う。
result = pd.read_sql(query, conn)
print(result)

conn.close()

   id     name  age
0   3  Charlie   35


### 行の追加

In [4]:
conn = sqlite3.connect(':memory:')
df.to_sql('df', conn, index=False, if_exists='replace')

# 修正：文字列はクォーテーションで囲む
query = """INSERT INTO df ('id', 'name', 'age')
            VALUES (4, 'Jhone', 18)
            """


# INSERTなどデータ変更系はpd.read_sqlではなくconn.executeを使います
conn.execute(query)
conn.commit()

# 挿入後のデータを確認
result = pd.read_sql("SELECT * FROM df", conn)
print(result)
conn.close()

   id     name  age
0   1    Alice   25
1   2      Bob   30
2   3  Charlie   35
3   4    Jhone   18


### 空の列の追加

In [5]:
conn = sqlite3.connect(':memory:')
df.to_sql('df', conn, index=False, if_exists='replace')
query = """ALTER TABLE df ADD COLUMN SEX TEXT"""

conn.execute(query)
conn.commit()

# その他はNULLのままなので、全体を確認
result = pd.read_sql("SELECT * FROM df", conn)
print(result)

conn.close()

   id     name  age   SEX
0   1    Alice   25  None
1   2      Bob   30  None
2   3  Charlie   35  None


### 列の追加２

In [6]:
'''列を追加した場合は値を１行毎に入れる必要がある
列を指定してリスト型で読み込ませることはSQLでは出来ない'''

'列を追加した場合は値を１行毎に入れる必要がある\n列を指定してリスト型で読み込ませることはSQLでは出来ない'

In [7]:
conn = sqlite3.connect(':memory:')
df.to_sql('df', conn, index=False, if_exists='replace')
query = """ALTER TABLE df 
            ADD COLUMN SEX TEXT;    
            
            UPDATE df 
                SET SEX = 'W' 
                    WHERE id = 1;
            UPDATE df 
                SET SEX = 'M' 
                    WHERE id = 2;
            UPDATE df 
                SET SEX = 'W' 
                    WHERE id = 3;"""

conn.executescript(query)
conn.commit()

# その他はNULLのままなので、全体を確認
result = pd.read_sql("SELECT * FROM df", conn)
print(result)

conn.close()

#TEXTはデータの型INTEGER（整数）,REAL(浮動小数点型), TEXT(文字型)

   id     name  age SEX
0   1    Alice   25   W
1   2      Bob   30   M
2   3  Charlie   35   W


In [70]:

data = {
    'シーン': [
        '集計結果を新規テーブルとして保存したい',
        '既存テーブルに新しい列を追加したい',
        '既存テーブルの行のデータを更新したい'
    ],
    '主な使い方': [
        'CREATE TABLE ... AS SELECT ... GROUP BY ...',
        'ALTER TABLE ... ADD COLUMN ...',
        'UPDATE ... SET ... WHERE ...'
    ]
}

df_summary = pd.DataFrame(data)
df_summary

,シーン,主な使い方
0,集計結果を新規テーブルとして保存したい,CREATE TABLE ... AS SELECT ... GROUP BY ...
1,既存テーブルに新しい列を追加したい,ALTER TABLE ... ADD COLUMN ...
2,既存テーブルの行のデータを更新したい,UPDATE ... SET ... WHERE ...


### 行の削除

In [8]:
conn = sqlite3.connect(':memory:')
df.to_sql('df', conn, index=False, if_exists='replace')
query = """DELETE
            FROM df
            where name = 'Bob'"""

conn.execute(query)
conn.commit()

# その他はNULLのままなので、全体を確認
result = pd.read_sql("SELECT * FROM df", conn)
print(result)

conn.close()

   id     name  age
0   1    Alice   25
1   3  Charlie   35


### 列の削除

In [9]:
'''列の削除はできない'''

'列の削除はできない'

### NaNがある行は削除する

In [14]:
df_n = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, np.nan, 35]
})
# np.nanをNoneに置換
df_n['age'] = df_n['age'].where(df_n['age'].notna(), None)
df_n

,id,name,age
0,1,Alice,25.0
1,2,Bob,NaN
2,3,Charlie,35.0


In [19]:
conn = sqlite3.connect(':memory:')
df_n.to_sql('people', conn, index=False, if_exists='replace')

# NULLがある行を削除
query = '''
DELETE FROM people
WHERE age IS NULL
'''

conn.execute(query)
conn.commit()

# 結果を確認
result = pd.read_sql("SELECT * FROM people", conn)
print(result)

conn.close()

   id     name   age
0   1    Alice  25.0
1   3  Charlie  35.0


### 含むのフィルタcontains

In [30]:
import sqlite3
import pandas as pd

df_contains = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Amanda', 'Charlie'],
    'age': [25, 30, 22, 35]
})
df_contains

,id,name,age
0,1,Alice,25
1,2,Bob,30
2,3,Amanda,22
3,4,Charlie,35


In [28]:
#nameにAとaを含むレコードを残す
conn = sqlite3.connect(':memory:')
df_contains.to_sql('df_contains', conn, index=False, if_exists='replace')

# name列に'A'を含む行だけ残すためにそれ以外を削除
query = '''
DELETE 
FROM df_contains 
WHERE name NOT LIKE '%A%'
'''

conn.execute(query)
conn.commit()

result = pd.read_sql("SELECT * FROM df_contains", conn)
print(result)

conn.close()

   id     name  age
0   1    Alice   25
1   3   Amanda   22
2   4  Charlie   35


In [32]:
#nameにAを含むレコードを残す
conn = sqlite3.connect(':memory:')
df_contains.to_sql('df_contains', conn, index=False, if_exists='replace')

# name列に'A'を含む行だけ残すためにそれ以外を削除
query = '''
DELETE 
FROM df_contains 
WHERE name NOT  GLOB '*A*'
'''

conn.execute(query)
conn.commit()

result = pd.read_sql("SELECT * FROM df_contains", conn)
print(result)

conn.close()

   id    name  age
0   1   Alice   25
1   3  Amanda   22


In [33]:
'''GLOBとLIKEを使い分ける必要がある'''

'GLOBとLIKEを使い分ける必要がある'

In [34]:
#該当文字を含んでいたらそのレコードを削除する

conn = sqlite3.connect(':memory:')
df_n.to_sql('people', conn, index=False, if_exists='replace')

# 小文字の'a'を含む名前のレコードを削除

query = '''
DELETE FROM people WHERE name GLOB '*a*'
'''

conn.execute("")
conn.commit()

result = pd.read_sql("SELECT * FROM people", conn)
print(result)

conn.close()

   id     name   age
0   1    Alice  25.0
1   2      Bob   NaN
2   3  Charlie  35.0


### 計算

In [35]:
df_cal = pd.DataFrame({
    'id': [1, 2, 3],
    'age': [30, 40, 50],
    'base_age': [10, 15, 20]
})
df_cal

,id,age,base_age
0,1,30,10
1,2,40,15
2,3,50,20


### 新規の列を追加して計算結果を格納

In [49]:
conn = sqlite3.connect(':memory:')
df_cal.to_sql('df_cal', conn, index=False, if_exists='replace')

# 複数のSQL文を一括実行しトランザクションとして処理
conn.executescript("""
    ALTER TABLE df_cal ADD COLUMN age_difference INTEGER;
    
    UPDATE 
    df_cal 
    SET age_difference = age - base_age;
""")
conn.commit()

# 結果を読み込み表示
result = pd.read_sql("SELECT * FROM df_cal", conn)
print(result)

conn.close()

   id  age  base_age  age_difference
0   1   30        10              20
1   2   40        15              25
2   3   50        20              30


### 既存の列を計算結果で更新

In [48]:
conn = sqlite3.connect(':memory:')
df_cal.to_sql('df_cal', conn, index=False, if_exists='replace')

# 複数のSQL文を一括実行しトランザクションとして処理
conn.executescript("""
    UPDATE df_cal SET base_age=age - base_age;
""")
conn.commit()

# 結果を読み込み表示
result = pd.read_sql("SELECT * FROM df_cal", conn)
print(result)

conn.close()

   id  age  base_age
0   1   30        20
1   2   40        25
2   3   50        30


### 新規の列を追加して平均値を算出し格納する

In [60]:
# SQLiteメモリDBに接続
conn = sqlite3.connect(':memory:')
df_cal.to_sql('df_cal', conn, index=False, if_exists='replace')


query = """
    ALTER TABLE df_cal ADD COLUMN avg_age REAL;
    UPDATE df_cal
    SET avg_age = (SELECT AVG(age) FROM df_cal);
"""

# 列を追加し、平均値で更新（executescriptで一括処理）
conn.executescript(query)
conn.commit()

# 結果を取得して表示
result = pd.read_sql("SELECT * FROM df_cal", conn)
print(result)

conn.close()

   id  age  base_age  avg_age
0   1   30        10     40.0
1   2   40        15     40.0
2   3   50        20     40.0


### 少数点を扱う計算

In [75]:
df_cal2 = pd.DataFrame({
    'id': [1, 2, 3],
    'data': [20.5, 4.78, 50.21],
    'base_value': [2.85, 10.78, 1.21],
})
df_cal2

,id,data,base_value
0,1,20.50,2.85
1,2,4.78,10.78
2,3,50.21,1.21


In [85]:
conn = sqlite3.connect(':memory:')
df_cal2.to_sql('df_cal2', conn, index=False, if_exists='replace')

query = """
ALTER TABLE df_cal2 ADD COLUMN value REAL;

UPDATE df_cal2
SET value = ROUND(data * 1.0 / base_value, 3);
"""

# 列を追加し、平均値で更新（executescriptで一括処理）
conn.executescript(query)
conn.commit()

# 結果を取得して表示
result = pd.read_sql("SELECT * FROM df_cal2", conn)
print(result)

conn.close()

   id   data  base_value   value
0   1  20.50        2.85   7.193
1   2   4.78       10.78   0.443
2   3  50.21        1.21  41.496


### 条件分岐

In [ ]:
'''
1つ判定結果カラムを作って
ageが１０以下であればjudgeカラムに１を書いて
ageが１５以上であればjudgeカラムに５を書いてほしい
それ以外は0を入れたい
'''

In [63]:
df_if = pd.DataFrame({
    'id': [1, 2, 3,4, 5, 6, 7],
    'age': [30, 40, 12, 5, 60, 100, 2],

})
df_if

,id,age
0,1,30
1,2,40
2,3,12
3,4,5
4,5,60
5,6,100
6,7,2


In [64]:
# SQLiteメモリDBに接続
conn = sqlite3.connect(':memory:')
df_if.to_sql('df_if', conn, index=False, if_exists='replace')

# judgeカラムを追加し、条件に応じて値を設定
query = """
ALTER TABLE df_if ADD COLUMN judge INTEGER;

UPDATE df_if
SET judge = CASE
    WHEN age <= 10 THEN 1
    WHEN age >= 15 THEN 5
    ELSE 0
END;
"""

conn.executescript(query)
conn.commit()

# 結果を取得して表示
result = pd.read_sql("SELECT * FROM df_if", conn)
print(result)

conn.close()

   id  age  judge
0   1   30      5
1   2   40      5
2   3   12      0
3   4    5      1
4   5   60      5
5   6  100      5
6   7    2      1


### データの結合

In [111]:
df1 = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35]
})
df1

,id,name,age
0,1,Alice,25
1,2,Bob,30
2,3,Charlie,35


In [112]:
df2 = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'SEX': ['w', 'm', 'm'],
    'Place': ['Hiroshima', 'Fukuoka', 'Yamaguchi']
})
df2

,id,name,SEX,Place
0,1,Alice,w,Hiroshima
1,2,Bob,m,Fukuoka
2,3,Charlie,m,Yamaguchi


In [113]:
df3 = pd.DataFrame({
    'id': [5, 10, 15],
    'name': ['Jhon', 'Mike', 'Lon'],
    'age': [15, np.nan , 32]
})
df3

,id,name,age
0,5,Jhon,15.0
1,10,Mike,NaN
2,15,Lon,32.0


### 横結合

In [114]:
'''df1とdf2を横結合してみる'''

'df1とdf2を横結合してみる'

In [115]:
# SQLiteメモリDBに接続
conn = sqlite3.connect(':memory:')

# データフレームをSQLテーブルに書き込み
df1.to_sql('df1', conn, index=False, if_exists='replace')
df2.to_sql('df2', conn, index=False, if_exists='replace')

# SQLで内部結合 (id, nameをキーにJOIN)
query = """
SELECT df1.id, df1.name, df1.age, df2.SEX, df2.Place
FROM df1
INNER JOIN df2
ON df1.id = df2.id AND df1.name = df2.name
"""

# SQL実行してデータフレームに読み込み
result = pd.read_sql(query, conn)

print(result)

'''
INNER JOIN で両テーブルのidとnameが一致する行だけ結合
ON df1.id = df2.id AND df1.name = df2.nameで複数カラムによる結合条件を指定
必要なカラムだけ SELECT節で指定して取得可能
'''

   id     name  age SEX      Place
0   1    Alice   25   w  Hiroshima
1   2      Bob   30   m    Fukuoka
2   3  Charlie   35   m  Yamaguchi


'\nINNER JOIN で両テーブルのidとnameが一致する行だけ結合\nON df1.id = df2.id AND df1.name = df2.nameで複数カラムによる結合条件を指定\n必要なカラムだけ SELECT節で指定して取得可能\n'

In [116]:
import pandas as pd

# 結合の種類データを辞書で定義
join_types = {
    "結合の種類": ["内部結合", "左外部結合", "右外部結合", "完全外部結合", "クロス結合"],
    "英語名": ["INNER JOIN", "LEFT OUTER JOIN", "RIGHT OUTER JOIN", "FULL OUTER JOIN", "CROSS JOIN"],
    "説明": [
        "両方のテーブルで結合条件を満たす行だけを取得。",
        "左側のテーブルの全行を取得し、右側テーブルに結合条件を満たす行があれば結合。なければNULL。",
        "右側のテーブルの全行を取得し、左側テーブルに結合条件を満たす行があれば結合。なければNULL。",
        "両方のテーブルの全行を取得し、結合条件を満たすところは結合。満たさないところはNULLで埋める。",
        "両方のテーブルのすべての組み合わせの行を返す（直積）。"
    ],
    "結果のイメージ": [
        "両テーブルでキーが一致する行だけ",
        "左のテーブルの全行＋結合する右のデータ（無ければNULL）",
        "右のテーブルの全行＋結合する左のデータ（無ければNULL）",
        "両テーブルの全行＋結合できない箇所はNULL",
        "左テーブルの全行×右テーブルの全行"
    ],
}

# データフレーム作成
df_join_types = pd.DataFrame(join_types)

# 表示
df_join_types

,結合の種類,英語名,説明,結果のイメージ
0,内部結合,INNER JOIN,両方のテーブルで結合条件を満たす行だけを取得。,両テーブルでキーが一致する行だけ
1,左外部結合,LEFT OUTER JOIN,左側のテーブルの全行を取得し、右側テーブルに結合条件を満たす行があれば結合。なければNULL。,左のテーブルの全行＋結合する右のデータ（無ければNULL）
2,右外部結合,RIGHT OUTER JOIN,右側のテーブルの全行を取得し、左側テーブルに結合条件を満たす行があれば結合。なければNULL。,右のテーブルの全行＋結合する左のデータ（無ければNULL）
3,完全外部結合,FULL OUTER JOIN,両方のテーブルの全行を取得し、結合条件を満たすところは結合。満たさないところはNULLで埋める。,両テーブルの全行＋結合できない箇所はNULL
4,クロス結合,CROSS JOIN,両方のテーブルのすべての組み合わせの行を返す（直積）。,左テーブルの全行×右テーブルの全行


### 縦結合

In [117]:
# DB接続
conn = sqlite3.connect(':memory:')

# テーブル作成
df1.to_sql('df1', conn, index=False, if_exists='replace')
df3.to_sql('df3', conn, index=False, if_exists='replace')

# UNION ALLで縦結合
query = """
SELECT id, name, age FROM df1
UNION ALL
SELECT id, name, age FROM df3
"""

result = pd.read_sql(query, conn)

print(result)

conn.close()
# もとになるテーブルを読み込む
# コンキャットされる側をUNION ALL の後にセレクトで指定

   id     name   age
0   1    Alice  25.0
1   2      Bob  30.0
2   3  Charlie  35.0
3   5     Jhon  15.0
4  10     Mike   NaN
5  15      Lon  32.0


In [118]:
'''結合を行うケースでは*で省略するのは推奨されない。指定して書き込む必要がある'''

'結合を行うケースでは*で省略するのは推奨されない。指定して書き込む必要がある'

### グループ化

In [66]:
# サンプルデータを作成
df_gr = pd.DataFrame({
    'category': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'C', 'B'],
    'value': [10, 20, 10, 30, 20, 15, 30, 40, 25]
})
df_gr

,category,value
0,A,10
1,B,20
2,A,10
3,C,30
4,B,20
5,A,15
6,C,30
7,C,40
8,B,25


In [68]:
# SQLiteメモリDBに接続
conn = sqlite3.connect(':memory:')
df_gr.to_sql('df_gr', conn, index=False, if_exists='replace')

# SQLでcategoryごとに行数をカウント（グループ化）
query = """
SELECT 
    category,
    COUNT(*) AS count_per_category
FROM df_gr
GROUP BY category;
"""

result = pd.read_sql(query, conn)
print(result)

conn.close()

  category  count_per_category
0        A                   3
1        B                   3
2        C                   3


In [86]:
'''
GROUP BYの場合は新規でテーブルを作成するため
ALTER TABLEではなくSELECTで宣言する
'''

'\nGROUP BYの場合は新規でテーブルを作成するため\nALTER TABLEではなくSELECTで宣言する\n'

### with構文

In [ ]:
WITH
cte1 AS (
  SELECT * FROM table1 WHERE condition1
),
cte2 AS (
  SELECT * FROM table2 WHERE condition2
),
cte3 AS (
  SELECT cte1.id, cte2.value
  FROM cte1
  JOIN cte2 ON cte1.id = cte2.id
)
SELECT * FROM cte3 WHERE some_column > 100;

In [87]:
'''ただのDFみたいなもの'''

'ただのDFみたいなもの'